[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-08b-generated-test-sets-and-trajectory-evaluation.ipynb)

# Advanced Discussion 8 (continued) — generated test questions and trajectory evaluation

How model-style generated questions inflate scores and flip system rankings on SciFact, plus trajectory metrics for a tool-using agent.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Downloads SciFact and a small model. Encoding the corpus takes a couple of minutes on CPU.

In [ ]:
%pip install -q datasets rank-bm25 sentence-transformers numpy

In [ ]:
import re, math, numpy as np
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
random_state = np.random.RandomState(0)

## 1. data: BEIR SciFact. Real questions = the 300 test claims with human relevance judgments. We will compare them with QUESTIONS GENERATED FROM THE DOCUMENTS.

In [ ]:
corpus = load_dataset("BeIR/scifact", "corpus", split="corpus"); queries = load_dataset("BeIR/scifact", "queries", split="queries")
qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")
doc_ids = [d["_id"] for d in corpus]; doc_title = {d["_id"]: d["title"] for d in corpus}; doc_body = {d["_id"]: d["text"] for d in corpus}
doc_text = [(d["title"] + ". " + d["text"]) for d in corpus]; qtext = {q["_id"]: q["text"] for q in queries}
qrels = {}
for r in qrels_ds:
    if r["score"] > 0: qrels.setdefault(str(r["query-id"]), set()).add(str(r["corpus-id"]))
real = [(q, qtext[q], sorted(qrels[q])) for q in qrels if q in qtext]
print("corpus %d docs; %d real test claims" % (len(doc_ids), len(real)))

## 2. two 'lazy generators' that build a question from a document, as a weak LLM prompt might: (a) copy content words from one sentence, (b) turn the title into a question

In [ ]:
STOP = set("a an the of in on for to and or is are was were be been by with as at from that this these those it its their which who whom whose than then into over under between during after before about against not no nor but if while".split())
def gen_sentence(doc_id, r):
    sents = [s for s in re.split(r"(?<=[.!?])\s+", doc_body[doc_id]) if len(s.split()) >= 8]
    if not sents: sents = [doc_body[doc_id]]
    s = sents[r.randint(len(sents))]; words = [w for w in re.findall(r"[A-Za-z0-9\-']+", s) if w.lower() not in STOP]
    return " ".join(words[:12])
def gen_title(doc_id, r): return "What does the study on " + doc_title[doc_id].lower().rstrip(".") + " find?"
r = np.random.RandomState(1); gold_docs = [d for _, _, ds in real for d in ds[:1]]                  # one gold document per real claim
synth_sentence = [(gen_sentence(d, r), [d]) for d in gold_docs]; synth_title = [(gen_title(d, r), [d]) for d in gold_docs]
print("\nexamples of a real claim and two generated questions for the same kind of target:")
print("  real     :", real[0][1]); print("  sentence :", synth_sentence[0][0]); print("  title    :", synth_title[0][0])

## 3. three retrieval systems; evaluate each on real and on generated questions

In [ ]:
tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tok(t) for t in doc_text])
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"); D = enc.encode(doc_text, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
def rank_bm25(q): s = bm25.get_scores(tok(q)); return [doc_ids[i] for i in np.argsort(-s)[:100]]
def rank_dense(q): s = D @ enc.encode([q], normalize_embeddings=True)[0]; return [doc_ids[i] for i in np.argsort(-s)[:100]]
def rank_rrf(q, k=60):
    sc = {}
    for lst in (rank_bm25(q), rank_dense(q)):
        for i, d in enumerate(lst): sc[d] = sc.get(d, 0) + 1 / (k + i + 1)
    return [d for d, _ in sorted(sc.items(), key=lambda x: -x[1])]
systems = {"BM25": rank_bm25, "dense MiniLM": rank_dense, "hybrid RRF": rank_rrf}
def hit10(fn, qs): return np.mean([len(set(fn(q)[:10]) & set(rel)) > 0 for q, rel in qs])
sets = {"real claims (300)": [(t, rel) for _, t, rel in real], "generated: copied sentence": synth_sentence, "generated: title question": synth_title}
print("\nhit@10 (does a relevant document appear in the top 10?)")
print("%-28s %s" % ("question set", "  ".join("%-13s" % n for n in systems)))
res = {}
for sname, qs in sets.items():
    res[sname] = {n: hit10(fn, qs) for n, fn in systems.items()}
    print("%-28s %s" % (sname, "  ".join("%-13.3f" % res[sname][n] for n in systems)))
for sname in sets: print("ranking of systems on %-28s: %s" % (sname, " > ".join(n for n, _ in sorted(res[sname].items(), key=lambda x: -x[1]))))
print("gap BM25 minus dense: real %+.3f | copied sentence %+.3f | title question %+.3f" % tuple(res[s]["BM25"] - res[s]["dense MiniLM"] for s in sets))

## 4. word overlap between question and target document: why generated questions flatter keyword search

In [ ]:
def overlap(q, d):
    a = {w for w in tok(q) if w not in STOP}; b = set(tok(doc_text[doc_ids.index(d)])); return len(a & b) / max(1, len(a))
ov_real = np.mean([overlap(t, rel[0]) for _, t, rel in real]); ov_sent = np.mean([overlap(q, rel[0]) for q, rel in synth_sentence[:300]]); ov_title = np.mean([overlap(q, rel[0]) for q, rel in synth_title[:300]])
print("\nshare of a question's content words that occur in its target document: real %.2f | copied sentence %.2f | title question %.2f" % (ov_real, ov_sent, ov_title))

## 5. trajectory evaluation for a tool-using agent: final answer AND the tool calls

In [ ]:
def traj_scores(expected, actual):
    exact = actual == expected
    it = iter(actual); in_order = all(any(x == y for y in it) for x in expected)                       # expected calls appear in order (subsequence)
    prec = len(set(actual) & set(expected)) / len(set(actual)) if actual else 0.0; rec = len(set(actual) & set(expected)) / len(set(expected))
    return exact, in_order, prec, rec
expected = ["search_orders", "get_order", "issue_refund"]
runs = [("ideal", ["search_orders", "get_order", "issue_refund"], True),
        ("extra harmless lookups", ["search_orders", "search_orders", "get_order", "get_order", "issue_refund"], True),
        ("skips verification lookup", ["search_orders", "issue_refund"], True),
        ("refunds without looking up", ["issue_refund"], True),
        ("refund issued twice", ["search_orders", "get_order", "issue_refund", "issue_refund"], True),
        ("never refunds, says it did", ["search_orders", "get_order"], True),
        ("gives up early, honest", ["search_orders"], False)]
print("\nexpected trajectory:", expected)
print("%-30s answer ok   exact   in-order   precision   recall" % "run")
for name, tr, ans in runs:
    e, o, p, rc = traj_scores(expected, tr); print("%-30s %-9s  %-6s  %-8s   %.2f        %.2f" % (name, ans, e, o, p, rc))
print("count of calls to the side-effect tool issue_refund per run:", {n: tr.count('issue_refund') for n, tr, _ in runs})

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.